# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/venkat-vipul/flyrank_ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [24]:
!pip -q install duckdb pandas pyarrow huggingface_hub fsspec scikit-learn

In [25]:
from google.colab import userdata
from huggingface_hub import login
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

con = duckdb.connect()

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("Hugging Face and DuckDB authenticated successfully.")

Hugging Face and DuckDB authenticated successfully.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method Choice and Why

For this task, I selected **Logistic Regression** and **Random Forest** to predict whether a content item is declining (`is_declining_label`).

Logistic Regression serves as a simple and interpretable baseline classifier. It is fast to train, easy to explain, and provides a good reference point for evaluating more complex models.

Random Forest was chosen because it can capture non-linear relationships and interactions between features without requiring extensive feature engineering. It is also more robust to noise and provides feature importance scores, which help interpret the model's predictions.

To ensure an honest evaluation, the models are trained using the same train/test split and compared using the same evaluation metrics. Columns that directly define the target (`trend_direction` and `trend_pct`) are excluded from the feature set to prevent data leakage, as recommended in the project documentation.

In [26]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*



The target variable is `is_declining_label`, which is created from `trend_direction`. To prevent data leakage, the columns `trend_direction` and `trend_pct` are removed before training because they directly define the target.

A grouped train/test split is used based on `client_id`. This ensures that pages from the same client do not appear in both the training and testing sets, providing a more realistic evaluation of model performance on unseen clients.

The dataset is preprocessed using a pipeline that imputes missing values, scales numerical features for Logistic Regression, and one-hot encodes categorical features.

In [27]:
df = pd.read_csv("content_refresh_anonymized.csv")

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

drop_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "provider_used",
    "model_used",
]

X = df.drop(columns=drop_columns + ["is_declining_label"])
y = df["is_declining_label"]
groups = df["client_id"]

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print(f"Positive class (train): {y_train.mean():.2%}")
print(f"Positive class (test): {y_test.mean():.2%}")

Training samples: 23837
Testing samples: 6163
Positive class (train): 55.01%
Positive class (test): 51.10%


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The Week 4 rule-based baseline was recreated on the same dataset using the same scoring logic based on impressions, CTR, and average search position. Logistic Regression and Random Forest were trained using the same grouped train/test split and evaluated with the same metrics to determine whether supervised learning improves upon the rule-based approach.

Logistic Regression was selected as a simple and interpretable model, while Random Forest was chosen because it can capture more complex relationships between features. All models were evaluated on the same test set to ensure a fair comparison.

In [28]:
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

lr_pred = lr_pipeline.predict(X_test)
rf_pred = rf_pipeline.predict(X_test)

baseline_pred = (
    (X_test["impressions_90d"] >= 100) &
    (X_test["ctr"] < 2) &
    (X_test["avg_position"].between(5, 20))
).astype(int)

def evaluate_model(name, y_true, y_pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, zero_division=0)
    }

results = pd.DataFrame([
    evaluate_model("Week 4 Baseline", y_test, baseline_pred),
    evaluate_model("Logistic Regression", y_test, lr_pred),
    evaluate_model("Random Forest", y_test, rf_pred)
])

results.round(4)

,Model,Accuracy,Precision,Recall,F1 Score
0,Week 4 Baseline,0.5239,0.5443,0.4195,0.4738
1,Logistic Regression,0.7462,0.7260,0.8085,0.7650
2,Random Forest,0.8207,0.7969,0.8711,0.8323


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Random Forest achieved the best overall performance, followed by Logistic Regression. Both supervised learning models outperformed the Week 4 baseline, indicating that they were better at identifying declining content.

The Week 4 baseline used simple threshold-based rules based on impressions, CTR, and average search position. While easy to understand and implement, it could not capture more complex patterns in the data. Random Forest learned these relationships more effectively, resulting in the highest Accuracy, Precision, Recall, and F1-score.

The confusion matrix shows that most predictions were classified correctly, while the remaining errors likely reflect content influenced by factors not fully captured by the available features. Feature importance scores indicate that recent impressions, historical impressions, average position, and content age were among the strongest predictors of declining content.

In [29]:
rf_model = rf_pipeline.named_steps["classifier"]

feature_names = preprocessor.get_feature_names_out()

importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

print(classification_report(y_test, rf_pred))

cm = confusion_matrix(y_test, rf_pred)
print(cm)

print(importances.head(10))

errors = X_test.copy()
errors["Actual"] = y_test.values
errors["Predicted"] = rf_pred

errors = errors[errors["Actual"] != errors["Predicted"]]

errors.head(10)

              precision    recall  f1-score   support

           0       0.85      0.77      0.81      3014
           1       0.80      0.87      0.83      3149

    accuracy                           0.82      6163
   macro avg       0.82      0.82      0.82      6163
weighted avg       0.82      0.82      0.82      6163

[[2315  699]
 [ 406 2743]]
                       Feature  Importance
18   num__impressions_prev_30d    0.169322
15   num__impressions_last_30d    0.130380
5         num__impressions_90d    0.067555
25           num__avg_position    0.053078
13  num__days_with_impressions    0.051008
21       num__content_age_days    0.040847
17      num__sessions_last_30d    0.028140
4              num__char_count    0.026571
24                    num__ctr    0.025458
3              num__word_count    0.025372


,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,impressions_90d,clicks_90d,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,Actual,Predicted
13,10.0,0.00,LOW,0.00,keyword article,informational,1342.0,8469.0,307,0,...,8000-15000,0.00,39.8,0.00,25.00,0.0,moderate,page_3_5,0,1
26,0.0,0.00,LOW,0.00,keyword article,informational,2686.0,17181.0,2426,3,...,15000-25000,0.12,30.0,0.00,11.11,0.0,moderate,page_3_5,0,1
36,0.0,0.00,LOW,0.00,keyword article,informational,2510.0,15518.0,371,5,...,15000-25000,1.35,5.4,0.00,0.00,0.0,moderate,page_1,0,1
81,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,320,1,...,NaN,0.31,9.6,0.00,0.00,20.0,moderate,page_1,1,0
129,30.0,0.00,LOW,0.00,keyword article,informational,2396.0,15734.0,2159,1,...,15000-25000,0.05,21.3,0.00,0.00,0.0,moderate,page_3_5,1,0
148,NaN,NaN,NaN,NaN,keyword article,commercial,NaN,NaN,41650,28,...,NaN,0.07,4.4,2.38,3.92,0.0,excellent,page_1,1,0
165,10.0,0.08,LOW,0.05,keyword article,informational,2883.0,18118.0,1035,0,...,15000-25000,0.00,23.6,0.00,0.00,0.0,moderate,page_3_5,1,0
204,1000.0,0.11,LOW,2.33,keyword article,informational,2873.0,18718.0,5996,6,...,15000-25000,0.10,19.0,11.11,11.11,0.0,good,striking,0,1
252,20.0,0.05,LOW,0.00,keyword article,commercial,NaN,NaN,1334,0,...,NaN,0.00,50.1,0.00,50.00,0.0,moderate,deep,1,0
277,40.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,7915,14,...,NaN,0.18,10.7,3.57,7.14,0.0,good,striking,0,1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.